In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from nltk.stem import WordNetLemmatizer
from google.colab import drive
from joblib import dump, load

import pandas as pd
import numpy as np
import nltk
import re

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
class EmailClassifier:
  def __init__(self):
    self.transformer = TfidfVectorizer()
    self.stemmer = WordNetLemmatizer()
    self.forest = RandomForestClassifier(n_estimators = 500, criterion = "gini", max_depth = 9)

    self.data = pd.read_csv('/content/drive/MyDrive/phishing.csv', encoding = 'ISO-8859-1')
    self.data = self.data.loc[:, ~self.data.columns.str.contains('^Unnamed')]

    self.x = self.data["Email Text"][0:5000]
    self.y = self.data["Email Type"][0:5000]

  def get_best_params(self, should_apply = False, refit = False):
    params = {
        "criterion": ("gini", "entropy", "log_loss"),
        "n_estimators": (250, 500, 750, 1000),
        "max_depth": (5, 7, 9)
    }
    grid = GridSearchCV(RandomForestClassifier(), params, verbose = 3, n_jobs = 7, cv = 3)
    grid.fit(self.x_train, self.y_train)

    if should_apply:
      self.forest.criterion = grid.best_params_.criterion
      self.forest["n_estimators"] = grid.best_params_["n_estimators"]
      self.forest["max_depth"] = grid.best_params_["max_depth"]

      if refit:
        self.fit()
    return grid.best_params_

  def __clean_documents(self, documents):
    cleanedDocuments = []

    for index in range(len(documents)):
      document = documents[index]

      document = re.sub(r'\W', ' ', str(documents[index]))
      document = re.sub(r'\s+[a-zA-Z]\s+', ' ', document)
      document = re.sub(r'\^[a-zA-Z]\s+', ' ', document)
      document = re.sub(r'\s+', ' ', document, flags = re.I)
      document = re.sub(r'^b\s+', '', document)

      document = document.lower()
      document = document.split()

      document = [self.stemmer.lemmatize(word) for word in document]
      document = ' '.join(document)

      cleanedDocuments.append(document)
    return cleanedDocuments

  def __prep_data(self, data_to_prep, is_predict = False):
    prepped_data = self.__clean_documents(data_to_prep)

    if not is_predict:
      prepped_data = self.transformer.fit_transform(prepped_data).toarray()
    else:
      prepped_data = self.transformer.transform(prepped_data).toarray()

    return prepped_data

  def start(self):
    self.x = self.__prep_data(self.x)
    self.x_train, self.x_test, self.y_train, self.y_test = train_test_split(self.x, self.y, test_size = 0.3, stratify = self.y)

  def fit(self):
    self.forest.fit(self.x_train, self.y_train)

  def predict(self):
    pred = self.forest.predict(self.x_test)
    self.score = accuracy_score(pred, self.y_test)
    return pred

  def is_malicious(self, messages):
    messages = self.__prep_data(messages, True)
    return self.forest.predict(messages)

In [ ]:
try:
  model = load('/content/drive/MyDrive/Models/__emailModel.joblib')
except:
  model = EmailClassifier()
  model.start()
  model.fit()

  dump(model, '/content/drive/MyDrive/Models/__emailModel.joblib')

In [ ]:
malicious_email_test = """
Protect your financial well-being.
 Purchase an Extended Auto Warranty for your Car today. CLICK HERE for a FREE no obligation quote.
 http://www.takemetothesavings.com/warranty/

 Car troubles always seem to happen at the worst possible time. Protect yourself and your family with a quality Extended Warranty for your car, truck, or SUV, so that a large expense cannot hit you all at once. We cover most vehicles with less than 150,000 miles.

 Buy DIRECT! Our prices are 40-60% LESS!

 We offer fair prices and prompt, toll-free claims service.  Get an Extended Warranty on your car today.

 Warranty plan also includes:

 1)   24-Hour Roadside Assistance.
 2)   Rental Benefit.
 3)   Trip Interruption Intervention.
 4)   Extended Towing Benefit.

 CLICK HERE for a FREE no obligation quote.
 http://www.takemetothesavings.com/warranty/





 ---------------------------------------
 To unsubscribe, go to:
 http://www.takemetothesavings.com/stopthemailplease/
 Please allow 48-72 hours for removal.
 """

In [ ]:
safe_email_test = """
Matthew French wrote:
>>Happiness was restored to the world with gcc 2.95 and later.
> I do not have the time to follow the compiler "wars", but I notice that I
> must use egcs to build 64 bit SPARC code.2.95 was where the reintegration effort started and work on the other
projects fell away. 3.0 was the target for completion of that work.> GCC 3 should do it, but because it is so "buggy"[1] it is not worth trying
> to use unless you really want to track down those compiler errors... :(A lot of those problems seem to have been shaken out by Redhat's ballsy
gcc 2.96 stunt. I've been trying out RedHat Limbo for a few weeks now,
equipped with gcc 3.1. I haven't fallen foul of compiler issue so far,
that I'm aware of anyhow. That said I'm glad I don't do much C++ - seems
pretty much every version of gcc (2.95, 2.96, 3.0, 3.1, and the
forthcoming 3.2) breaks C++ binary compatibility in some way or other.Paul.
--
Irish Linux Users' Group: ilug@linux.ie
http://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.
List maintainer: listmaster@linux.ie

"""

In [ ]:
model.is_malicious([malicious_email_test, safe_email_test])

array(['Phishing Email', 'Safe Email'], dtype=object)